In [ ]:
# 03-latest (2024-12-01) - Fixed typing_extensions/mlflow compatibility issue
# Install compatible versions to avoid "cannot import name 'Sentinel'" error
%pip install typing_extensions>=4.5.0 mlflow>=2.9.0 -q

# Restart Python to pick up the new packages
dbutils.library.restartPython()

In [ ]:
# SP-3: Enhanced Opponent Modeling
# Using sklearn on sampled data - SparkML has 100MB model limit on Serverless
# that applies to the ENTIRE ML operation, not just model size
#
# ENHANCEMENTS IMPLEMENTED:
# - K-Fold Cross-Validation (StratifiedKFold)
# - Hyperparameter Tuning (RandomizedSearchCV)
# - Learning Curves (for bias/variance diagnosis)
# - Probability Calibration (CalibratedClassifierCV for RF)
# - Confusion Matrices with Seaborn heatmaps
# - class_weight='balanced' for class imbalance

import time
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    cross_val_score, StratifiedKFold, cross_validate, learning_curve
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report, roc_curve, auc, roc_auc_score, confusion_matrix
)
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import matplotlib.pyplot as plt
import seaborn as sns

# Set up MLflow experiment
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/leo.lwakabamba@gmail.com/poker-ml-opponent-modeling")

print("=" * 80)
print("SP-3: Enhanced Opponent Modeling (sklearn on sampled data)")
print("=" * 80)
print("NOTE: SparkML on Serverless has 100MB limit on entire ML operation.")
print("      Using sklearn with strategic sampling instead.")
print(f"[MLflow] Experiment set")
start_time = time.time()

spark = SparkSession.builder.getOrCreate()
print(f"Spark version: {spark.version}")

In [ ]:
# Configuration
UC_VOLUME_DIR = '/Volumes/pokerml/default/data/'

# Input: SP-2 output
INPUT_PATH = UC_VOLUME_DIR + 'processed/sp2_player_events_labeled'

# Output: Models
MODELS_DIR = UC_VOLUME_DIR + 'models/'

# MLflow Model Registry - use Unity Catalog three-level namespace
MODEL_REGISTRY_PREFIX = "pokerml.default"  # catalog.schema

STREETS = ['preflop', 'flop', 'turn', 'river']
TEST_SIZE = 0.2
RANDOM_STATE = 42

# ============================================================================
# DEBUG MODE - Read from pipeline config (UC Volume - persists across Python restarts)
# ============================================================================
import json

# UC Volume path (persists across restarts, unlike /tmp)
CONFIG_PATH = '/Volumes/pokerml/default/data/pipeline_config.json'

try:
    with open(CONFIG_PATH, 'r') as f:
        config = json.load(f)
    DEBUG_MODE = config.get('debug_mode', True)
    MAX_ROWS = config.get('max_rows', 10000)
    # NEW: Use training_sample_size for model training notebooks (SP-03 and SP-05)
    TRAINING_SAMPLE_SIZE = config.get('training_sample_size', 300000)
    print(f"   Loaded config from {CONFIG_PATH}")
    print(f"   DEBUG_MODE={DEBUG_MODE}, MAX_ROWS={MAX_ROWS:,}, TRAINING_SAMPLE_SIZE={TRAINING_SAMPLE_SIZE:,}")
except FileNotFoundError:
    DEBUG_MODE = True
    MAX_ROWS = 10000
    TRAINING_SAMPLE_SIZE = 300000
    print(f"   Config not found at {CONFIG_PATH}, using defaults")

DEBUG_MAX_SAMPLES_PER_STREET = 5000  # Use 5k samples per street in debug mode
# ============================================================================

# Sample size per street for sklearn (fits in memory)
# In debug mode: use DEBUG_MAX_SAMPLES_PER_STREET
# In full mode: use TRAINING_SAMPLE_SIZE (300k) NOT MAX_ROWS (7.5M)
# This allows other notebooks (SP-06-09) to use full data while SP-03/SP-05 train on smaller sample
MAX_SAMPLES_PER_STREET = DEBUG_MAX_SAMPLES_PER_STREET if DEBUG_MODE else TRAINING_SAMPLE_SIZE

# Position features - Critical for opponent modeling
# position_from_button: 0=BTN, 1=SB, 2=BB, 3=UTG, etc.
# num_players: table size context
POSITION_NUMERIC = ['position_from_button', 'num_players']

BASE_NUMERIC = [
    'pot_size', 'amount', 'action_no_in_hand', 'raises_so_far', 'calls_so_far',
    'starting_stack', 'stack_vs_table_median',
    'vpip_last3_hist', 'vpip_last5_hist', 'vpip_last10_hist',
    'pfr_last3_hist', 'pfr_last5_hist', 'pfr_last10_hist',
    'agg_factor_last3_hist', 'agg_factor_last5_hist', 'agg_factor_last10_hist',
    'street_adv_last3_hist', 'street_adv_last5_hist', 'street_adv_last10_hist',
    'stack_trend_last3_hist', 'stack_trend_last5_hist', 'stack_trend_last10_hist'
]

BET_NUMERIC = ['bet_pct_pot']
BOARD_NUMERIC = ['board_pair_or_better', 'board_flush_possible', 'board_straight_possible']

# Position features are especially important for preflop decisions
FEATURES_BY_STREET = {
    'preflop': POSITION_NUMERIC + BASE_NUMERIC + BET_NUMERIC,
    'flop': POSITION_NUMERIC + BASE_NUMERIC + BET_NUMERIC + BOARD_NUMERIC,
    'turn': POSITION_NUMERIC + BASE_NUMERIC + BET_NUMERIC + BOARD_NUMERIC,
    'river': POSITION_NUMERIC + BASE_NUMERIC + BET_NUMERIC + BOARD_NUMERIC,
}

print(f"Input path: {INPUT_PATH}")
print(f"Models directory: {MODELS_DIR}")
print(f"Model Registry: {MODEL_REGISTRY_PREFIX}")
print(f"Max samples per street: {MAX_SAMPLES_PER_STREET:,}")
if DEBUG_MODE:
    print(f"\n*** DEBUG MODE ENABLED ***")
else:
    print(f"\n*** FULL MODE - using TRAINING_SAMPLE_SIZE={TRAINING_SAMPLE_SIZE:,} (not MAX_ROWS) ***")
print(f"\nFeature groups:")
print(f"   Position: {POSITION_NUMERIC}")
print(f"   Base numeric: {len(BASE_NUMERIC)} features")
print(f"   Bet sizing: {BET_NUMERIC}")
print(f"   Board texture: {BOARD_NUMERIC}")

In [ ]:
# Load data using Spark, then sample per street for sklearn
print("\n[1/4] Loading and sampling data...")

print(f"   Loading from: {INPUT_PATH}")

# Load full dataset with Spark
spark_df = spark.read.parquet(INPUT_PATH)
total_rows = spark_df.count()
print(f"   Total rows in dataset: {total_rows:,}")

# ============================================================================
# SELECT TRACE_HAND_ID FOR PIPELINE VALIDATION
# ============================================================================
TRACE_HAND_ID = spark_df.select('hand_id').first()['hand_id']
print(f"\n*** TRACE_HAND_ID selected: {TRACE_HAND_ID} ***")

# ============================================================================
# TRACE: RAW INPUT DATA
# ============================================================================
print(f"\n[TRACE] RAW INPUT from SP-2 for {TRACE_HAND_ID}:")
spark_df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'action_type',
    'position_from_button', 'position_name', 'num_players',
    'bucket_label', 'hand_equity'
).show(20, truncate=False)

# Map bucket labels to 3 classes
spark_df = spark_df.withColumn(
    'label_3class',
    F.when(F.col('bucket_label') == 'air', 'air')
     .when(F.col('bucket_label') == 'nutted', 'nutted')
     .otherwise('middle')
)

# ============================================================================
# TRACE: AFTER LABEL MAPPING
# ============================================================================
print(f"\n[TRACE] AFTER label_3class mapping for {TRACE_HAND_ID}:")
spark_df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'actor', 'street', 'bucket_label', 'label_3class',
    'position_from_button', 'position_name'
).show(20, truncate=False)

# Sample and convert to pandas per street (memory efficient)
street_data = {}
for street in STREETS:
    street_spark = spark_df.filter(F.col('street') == street)
    street_count = street_spark.count()
    
    if street_count > MAX_SAMPLES_PER_STREET:
        sample_frac = MAX_SAMPLES_PER_STREET / street_count
        street_spark = street_spark.sample(fraction=sample_frac, seed=RANDOM_STATE)
    
    # Select only needed columns before converting to pandas
    features = FEATURES_BY_STREET[street]
    cols_to_select = features + ['label_3class', 'hand_id']
    available_cols = [c for c in cols_to_select if c in spark_df.columns]
    
    street_pandas = street_spark.select(available_cols).toPandas()
    street_data[street] = street_pandas
    print(f"   {street}: {len(street_pandas):,} samples (of {street_count:,})")
    
    # TRACE: Check if trace hand is in this street's data
    trace_in_street = street_pandas[street_pandas['hand_id'] == TRACE_HAND_ID]
    if len(trace_in_street) > 0:
        print(f"      [TRACE] {TRACE_HAND_ID} has {len(trace_in_street)} rows in {street}")

print(f"\n   Total samples loaded: {sum(len(df) for df in street_data.values()):,}")

# ============================================================================
# TRACE: VERIFY POSITION FEATURES IN LOADED DATA
# ============================================================================
print(f"\n[TRACE] Position feature coverage per street:")
for street in STREETS:
    df = street_data[street]
    if 'position_from_button' in df.columns:
        non_null = df['position_from_button'].notna().sum()
        print(f"   {street}: {non_null:,}/{len(df):,} rows with position_from_button ({100*non_null/len(df):.1f}%)")
    else:
        print(f"   {street}: position_from_button NOT in columns")

In [ ]:
# sklearn pipeline builder with hyperparameter tuning, learning curves, and probability calibration
# UPDATED: Now uses sklearn Pipeline to bundle scaler + model together
# UPDATED: Added class_weight='balanced' to handle severe class imbalance
# UPDATED: Added K-Fold Cross-Validation for more reliable performance estimates
# UPDATED: Added RandomizedSearchCV for hyperparameter tuning
# UPDATED: Added learning_curve for bias/variance diagnosis
# UPDATED: Added CalibratedClassifierCV for probability calibration
# UPDATED: Each hyperparameter trial is logged as separate MLflow run with model
# UPDATED: Feature importance and ROC curves logged to each model run
# UPDATED: Added GradientBoosting with conservative hyperparameters
print("\n[2/4] Setting up sklearn models with Pipeline (scaler + model bundled)...")

from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier

# ============================================================================
# TRAINING SPEED SETTINGS - CONSERVATIVE
# ============================================================================
# CONSERVATIVE (good balance of speed and thoroughness):
N_SPLITS = 3                    # 3-fold cross-validation
CV_RANDOM_STATE = 42
HYPERPARAM_CV_SPLITS = 2        # 2-fold for hyperparameter tuning
N_ITER_RANDOM_SEARCH = 8        # 8 random combinations (more thorough)
LEARNING_CURVE_POINTS = 5       # 5 points on learning curve
COMPUTE_RF_LEARNING_CURVE = True   # Compute learning curve for RF
COMPUTE_GB_LEARNING_CURVE = False  # Skip learning curve for GB (slow)
# ============================================================================

def calculate_all_metrics(y_true, y_pred, y_proba, label_encoder):
    """Calculate all classification metrics including ROC AUC."""
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
    }
    
    # Per-class F1 scores
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    for i, class_name in enumerate(label_encoder.classes_):
        if i < len(f1_per_class):
            metrics[f'f1_{class_name}'] = f1_per_class[i]
    
    # Calculate ROC AUC (multiclass OvR)
    try:
        if y_proba is not None and len(np.unique(y_true)) > 1:
            metrics['roc_auc_ovr'] = roc_auc_score(y_true, y_proba, multi_class='ovr', average='weighted')
            metrics['roc_auc_ovo'] = roc_auc_score(y_true, y_proba, multi_class='ovo', average='weighted')
    except Exception as e:
        print(f"      Warning: Could not calculate ROC AUC: {e}")
    
    return metrics

def plot_confusion_matrix_heatmap(y_true, y_pred, label_encoder, street, model_type):
    """Plot confusion matrix as heatmap."""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_encoder.classes_, 
                yticklabels=label_encoder.classes_,
                ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(f'Confusion Matrix - {street.upper()} ({model_type})', fontsize=14)
    plt.tight_layout()
    return fig

def plot_roc_curves(y_test, y_proba, label_encoder, street, model_type):
    """Plot ROC curves for multiclass classification."""
    n_classes = len(label_encoder.classes_)
    
    # Binarize the output
    y_test_bin = label_binarize(y_test, classes=range(n_classes))
    
    # Handle binary case
    if n_classes == 2:
        y_test_bin = np.column_stack([1 - y_test_bin, y_test_bin])
    
    # Compute ROC curve and ROC area for each class
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        try:
            fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
            roc_auc[i] = auc(fpr[i], tpr[i])
        except Exception as e:
            fpr[i] = [0, 1]
            tpr[i] = [0, 1]
            roc_auc[i] = 0.5
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    for i, color in zip(range(n_classes), colors):
        class_name = label_encoder.classes_[i] if i < len(label_encoder.classes_) else f'Class {i}'
        ax.plot(fpr[i], tpr[i], color=color, lw=2,
                label=f'{class_name} (AUC = {roc_auc[i]:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.500)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title(f'ROC Curves - {street.upper()} ({model_type})', fontsize=14)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig, roc_auc

def plot_feature_importance(classifier, feature_names, street, model_type):
    """Plot and return feature importance chart."""
    importances = None
    
    if hasattr(classifier, 'feature_importances_'):
        importances = classifier.feature_importances_
    elif hasattr(classifier, 'coef_'):
        importances = np.abs(classifier.coef_).mean(axis=0)
    
    if importances is None:
        return None, None
    
    # Validate lengths match
    if len(importances) != len(feature_names):
        min_len = min(len(feature_names), len(importances))
        feature_names = feature_names[:min_len]
        importances = importances[:min_len]
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(12, 8))
    top_n = min(20, len(importance_df))
    top_features = importance_df.head(top_n)
    
    y_pos = np.arange(top_n)
    ax.barh(y_pos, top_features['importance'].values, align='center', color='steelblue')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features['feature'].values)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'Feature Importance - {street.upper()} ({model_type})', fontsize=14)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    return fig, importance_df

def plot_learning_curve(estimator, X, y, cv, street, model_type, train_sizes=None):
    """Plot learning curve to diagnose bias/variance."""
    if train_sizes is None:
        train_sizes = np.linspace(0.1, 1.0, LEARNING_CURVE_POINTS)
    
    print(f"      Computing learning curve (this may take a moment)...")
    
    try:
        train_sizes_abs, train_scores, val_scores = learning_curve(
            estimator, X, y, cv=cv, n_jobs=-1,
            train_sizes=train_sizes,
            scoring='f1_macro',
            random_state=CV_RANDOM_STATE
        )
        
        train_mean = train_scores.mean(axis=1)
        train_std = train_scores.std(axis=1)
        val_mean = val_scores.mean(axis=1)
        val_std = val_scores.std(axis=1)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        ax.fill_between(train_sizes_abs, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
        ax.plot(train_sizes_abs, train_mean, 'o-', color='blue', label='Training score')
        
        ax.fill_between(train_sizes_abs, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
        ax.plot(train_sizes_abs, val_mean, 'o-', color='orange', label='Cross-validation score')
        
        ax.set_xlabel('Training Set Size', fontsize=12)
        ax.set_ylabel('F1 Macro Score', fontsize=12)
        ax.set_title(f'Learning Curve - {street.upper()} ({model_type})', fontsize=14)
        ax.legend(loc='lower right', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        final_gap = train_mean[-1] - val_mean[-1]
        if final_gap > 0.1:
            ax.annotate(f'Gap: {final_gap:.3f} (possible overfitting)', 
                       xy=(train_sizes_abs[-1], val_mean[-1]),
                       xytext=(train_sizes_abs[-1] * 0.7, val_mean[-1] - 0.1),
                       arrowprops=dict(arrowstyle='->', color='red'),
                       fontsize=10, color='red')
        
        plt.tight_layout()
        
        learning_metrics = {
            'lc_final_train_score': float(train_mean[-1]),
            'lc_final_val_score': float(val_mean[-1]),
            'lc_train_val_gap': float(final_gap),
            'lc_train_std': float(train_std[-1]),
            'lc_val_std': float(val_std[-1]),
        }
        
        return fig, learning_metrics
    
    except Exception as e:
        print(f"      Warning: Could not compute learning curve: {e}")
        return None, {}

def run_cross_validation(pipeline, X, y, cv, model_type):
    """Run K-Fold cross-validation and return mean ± std metrics."""
    scoring = {
        'accuracy': 'accuracy',
        'f1_macro': 'f1_macro',
        'f1_weighted': 'f1_weighted',
    }
    
    print(f"      Running {N_SPLITS}-fold cross-validation...")
    cv_results = cross_validate(
        pipeline, X, y, cv=cv, scoring=scoring, 
        return_train_score=True, n_jobs=-1
    )
    
    cv_metrics = {}
    for metric in ['accuracy', 'f1_macro', 'f1_weighted']:
        train_key = f'train_{metric}'
        test_key = f'test_{metric}'
        cv_metrics[f'cv_{metric}_mean'] = cv_results[test_key].mean()
        cv_metrics[f'cv_{metric}_std'] = cv_results[test_key].std()
        cv_metrics[f'cv_train_{metric}_mean'] = cv_results[train_key].mean()
        cv_metrics[f'cv_train_{metric}_std'] = cv_results[train_key].std()
    
    print(f"      CV Accuracy: {cv_metrics['cv_accuracy_mean']:.4f} ± {cv_metrics['cv_accuracy_std']:.4f}")
    print(f"      CV F1 Macro: {cv_metrics['cv_f1_macro_mean']:.4f} ± {cv_metrics['cv_f1_macro_std']:.4f}")
    print(f"      CV F1 Weighted: {cv_metrics['cv_f1_weighted_mean']:.4f} ± {cv_metrics['cv_f1_weighted_std']:.4f}")
    
    train_test_gap = cv_metrics['cv_train_f1_macro_mean'] - cv_metrics['cv_f1_macro_mean']
    if train_test_gap > 0.1:
        print(f"      ⚠️ Potential overfitting: Train-Test gap = {train_test_gap:.4f}")
    
    return cv_metrics

def format_params_for_name(params):
    """Format parameters into a short readable string for run names."""
    parts = []
    for k, v in params.items():
        short_key = k.split('__')[-1]
        if isinstance(v, float):
            parts.append(f"{short_key}={v:.2g}")
        else:
            parts.append(f"{short_key}={v}")
    return "_".join(parts)

def run_hyperparameter_tuning_with_logging(base_pipeline, param_grid, X, y, cv, model_type, street, feature_names, label_encoder):
    """Run hyperparameter tuning and log EACH trial as a separate MLflow run."""
    from sklearn.model_selection import ParameterSampler
    
    print(f"      Running hyperparameter tuning ({N_ITER_RANDOM_SEARCH} iterations)...")
    print(f"      Each trial will be logged as a separate MLflow run with model...")
    
    param_list = list(ParameterSampler(param_grid, n_iter=N_ITER_RANDOM_SEARCH, random_state=CV_RANDOM_STATE))
    sample_input = pd.DataFrame(X[:5].values if hasattr(X, 'values') else X[:5], columns=feature_names)
    
    trial_results = []
    best_score = -np.inf
    best_pipeline = None
    best_params = None
    
    for i, params in enumerate(param_list):
        pipeline = clone(base_pipeline)
        pipeline.set_params(**params)
        
        param_str = format_params_for_name(params)
        run_name = f"{street}_{model_type}_trial{i+1}_{param_str}"
        
        scoring = {'f1_macro': 'f1_macro', 'accuracy': 'accuracy'}
        cv_results = cross_validate(pipeline, X, y, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
        
        mean_score = cv_results['test_f1_macro'].mean()
        std_score = cv_results['test_f1_macro'].std()
        mean_accuracy = cv_results['test_accuracy'].mean()
        
        print(f"         Trial {i+1}/{N_ITER_RANDOM_SEARCH}: F1={mean_score:.4f}±{std_score:.4f} | {param_str}")
        
        with mlflow.start_run(run_name=run_name, nested=True):
            mlflow.log_param("street", street)
            mlflow.log_param("model_type", model_type)
            mlflow.log_param("trial_number", i + 1)
            mlflow.log_param("is_best", False)
            
            for k, v in params.items():
                mlflow.log_param(k, str(v))
            
            mlflow.log_metric("cv_f1_macro_mean", mean_score)
            mlflow.log_metric("cv_f1_macro_std", std_score)
            mlflow.log_metric("cv_accuracy_mean", mean_accuracy)
            mlflow.log_metric("cv_train_f1_macro_mean", cv_results['train_f1_macro'].mean())
            
            pipeline.fit(X, y)
            
            signature = infer_signature(sample_input, pipeline.predict(X[:5]))
            mlflow.sklearn.log_model(pipeline, artifact_path="model", signature=signature)
        
        trial_results.append({
            'params': params,
            'mean_score': mean_score,
            'std_score': std_score,
            'pipeline': pipeline
        })
        
        if mean_score > best_score:
            best_score = mean_score
            best_pipeline = pipeline
            best_params = params
    
    param_str = format_params_for_name(best_params)
    with mlflow.start_run(run_name=f"{street}_{model_type}_BEST_{param_str}", nested=True):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("is_best", True)
        for k, v in best_params.items():
            mlflow.log_param(k, str(v))
        mlflow.log_metric("cv_f1_macro_mean", best_score)
        
        signature = infer_signature(sample_input, best_pipeline.predict(X[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
    
    print(f"      Best params: {best_params}")
    print(f"      Best CV F1 Macro: {best_score:.4f}")
    
    tuning_results = {
        'best_score': best_score,
        'best_params': {k: str(v) for k, v in best_params.items()},
        'all_trials': [
            {'params': {k: str(v) for k, v in t['params'].items()}, 
             'cv_f1_macro_mean': t['mean_score'],
             'cv_f1_macro_std': t['std_score']}
            for t in trial_results
        ]
    }
    
    return best_pipeline, best_params, tuning_results

def train_sklearn_pipelines(X_train, X_test, y_train, y_test, street, feature_names):
    """
    Train multiple sklearn Pipelines (scaler + model bundled together).
    
    FEATURES:
    - K-Fold Cross-Validation for reliable performance estimates
    - RandomizedSearchCV for hyperparameter tuning
    - Learning curves for bias/variance diagnosis
    - CalibratedClassifierCV for probability calibration (RF)
    - class_weight='balanced' to handle class imbalance
    - Feature importance and ROC curves logged to each model run
    """
    
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)
    
    print(f"   Classes found: {le.classes_}")
    print(f"   Train class distribution: {dict(zip(le.classes_, np.bincount(y_train_encoded)))}")
    print(f"   Test class distribution: {dict(zip(le.classes_, np.bincount(y_test_encoded)))}")
    
    class_counts = np.bincount(y_train_encoded)
    max_count = class_counts.max()
    min_count = class_counts.min()
    imbalance_ratio = max_count / max(min_count, 1)
    if imbalance_ratio > 5:
        print(f"   ⚠️ SEVERE CLASS IMBALANCE: ratio = {imbalance_ratio:.1f}x")
        print(f"      Using class_weight='balanced' to compensate")
    
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)
    cv_tune = StratifiedKFold(n_splits=HYPERPARAM_CV_SPLITS, shuffle=True, random_state=CV_RANDOM_STATE)
    
    results = []
    learning_curve_figures = {}
    
    sample_input = pd.DataFrame(X_train[:5].values, columns=feature_names)
    
    # =========================================================================
    # Model 1: Logistic Regression Pipeline
    # =========================================================================
    print("\n   Training LogisticRegression Pipeline (with hyperparameter tuning)...")
    with mlflow.start_run(run_name=f"{street}_LogisticRegression_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("pipeline", "StandardScaler + LogisticRegression")
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_classes", len(le.classes_))
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("hyperparam_tuning", True)
        mlflow.log_param("n_iter_search", N_ITER_RANDOM_SEARCH)
        
        base_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                max_iter=500, random_state=RANDOM_STATE, n_jobs=-1,
                solver='saga', multi_class='multinomial', class_weight='balanced'
            ))
        ])
        
        param_grid = {
            'classifier__C': [0.01, 0.1, 1.0, 10.0],
            'classifier__penalty': ['l1', 'l2'],
        }
        
        best_pipeline, best_params, tuning_results = run_hyperparameter_tuning_with_logging(
            base_pipeline, param_grid, X_train, y_train_encoded, cv_tune, 'LogReg', street, feature_names, le
        )
        
        mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
        mlflow.log_metric("tuning_best_score", tuning_results['best_score'])
        mlflow.log_dict(tuning_results, "hyperparameter_tuning_results.json")
        
        cv_metrics = run_cross_validation(best_pipeline, X_train, y_train_encoded, cv, 'LogReg')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        lc_fig, lc_metrics = plot_learning_curve(best_pipeline, X_train, y_train_encoded, cv, street, 'LogReg')
        if lc_fig:
            mlflow.log_figure(lc_fig, f"learning_curve_{street}_logreg.png")
            for name, value in lc_metrics.items():
                mlflow.log_metric(name, value)
            learning_curve_figures[f'LogReg_{street}'] = lc_fig
            plt.close(lc_fig)
        
        best_pipeline.fit(X_train, y_train_encoded)
        y_pred = best_pipeline.predict(X_test)
        y_proba = best_pipeline.predict_proba(X_test)
        
        metrics = calculate_all_metrics(y_test_encoded, y_pred, y_proba, le)
        for name, value in metrics.items():
            mlflow.log_metric(f"test_{name}", value)
        
        # Log confusion matrix
        try:
            cm_fig = plot_confusion_matrix_heatmap(y_test_encoded, y_pred, le, street, 'LogReg')
            mlflow.log_figure(cm_fig, f"confusion_matrix_{street}_logreg.png")
            plt.close(cm_fig)
        except Exception as e:
            print(f"      Warning: Could not plot confusion matrix: {e}")
        
        # Log ROC curves
        try:
            roc_fig, roc_auc_dict = plot_roc_curves(y_test_encoded, y_proba, le, street, 'LogReg')
            mlflow.log_figure(roc_fig, f"roc_curves_{street}_logreg.png")
            plt.close(roc_fig)
        except Exception as e:
            print(f"      Warning: Could not plot ROC curves: {e}")
        
        # Log feature importance
        try:
            classifier = best_pipeline.named_steps['classifier']
            fi_fig, fi_df = plot_feature_importance(classifier, feature_names, street, 'LogReg')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_logreg.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_logreg.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        signature = infer_signature(sample_input, best_pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics}
        results.append({
            'pipeline': best_pipeline, 'label_encoder': le,
            'model_type': 'LogisticRegression', 'y_proba': y_proba, 
            'best_params': best_params, **all_metrics
        })
        print(f"      Test: F1={metrics['f1_weighted']:.3f}, Acc={metrics['accuracy']:.3f}, F1_macro={metrics['f1_macro']:.3f}")
    
    # =========================================================================
    # Model 2: Random Forest Pipeline
    # =========================================================================
    print("\n   Training RandomForest Pipeline (with hyperparameter tuning + calibration)...")
    with mlflow.start_run(run_name=f"{street}_RandomForest_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("pipeline", "StandardScaler + RandomForest + Calibration")
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_classes", len(le.classes_))
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("hyperparam_tuning", True)
        mlflow.log_param("probability_calibration", True)
        mlflow.log_param("n_iter_search", N_ITER_RANDOM_SEARCH)
        
        base_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', RandomForestClassifier(
                random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced'
            ))
        ])
        
        param_grid = {
            'classifier__n_estimators': [100, 150, 200],
            'classifier__max_depth': [8, 12, 15, None],
            'classifier__min_samples_split': [2, 5, 10],
            'classifier__min_samples_leaf': [1, 2, 4],
            'classifier__max_features': ['sqrt', 'log2'],
        }
        
        best_pipeline, best_params, tuning_results = run_hyperparameter_tuning_with_logging(
            base_pipeline, param_grid, X_train, y_train_encoded, cv_tune, 'RF', street, feature_names, le
        )
        
        mlflow.log_params({f"best_{k}": str(v) for k, v in best_params.items()})
        mlflow.log_metric("tuning_best_score", tuning_results['best_score'])
        mlflow.log_dict(tuning_results, "hyperparameter_tuning_results.json")
        
        cv_metrics = run_cross_validation(best_pipeline, X_train, y_train_encoded, cv, 'RF')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        lc_metrics = {}
        if COMPUTE_RF_LEARNING_CURVE:
            lc_fig, lc_metrics = plot_learning_curve(best_pipeline, X_train, y_train_encoded, cv, street, 'RF')
            if lc_fig:
                mlflow.log_figure(lc_fig, f"learning_curve_{street}_rf.png")
                for name, value in lc_metrics.items():
                    mlflow.log_metric(name, value)
                learning_curve_figures[f'RF_{street}'] = lc_fig
                plt.close(lc_fig)
        else:
            print("      Skipping RF learning curve (COMPUTE_RF_LEARNING_CURVE=False)")
        
        best_pipeline.fit(X_train, y_train_encoded)
        
        # Probability calibration
        print("      Applying probability calibration (isotonic)...")
        try:
            calibrated_pipeline = CalibratedClassifierCV(best_pipeline, method='isotonic', cv='prefit')
            calibrated_pipeline.fit(X_train, y_train_encoded)
            y_pred = calibrated_pipeline.predict(X_test)
            y_proba = calibrated_pipeline.predict_proba(X_test)
            mlflow.log_param("calibration_method", "isotonic")
            mlflow.log_param("calibration_applied", True)
            final_pipeline = calibrated_pipeline
        except Exception as e:
            print(f"      Warning: Calibration failed, using uncalibrated: {e}")
            y_pred = best_pipeline.predict(X_test)
            y_proba = best_pipeline.predict_proba(X_test)
            mlflow.log_param("calibration_applied", False)
            final_pipeline = best_pipeline
        
        metrics = calculate_all_metrics(y_test_encoded, y_pred, y_proba, le)
        for name, value in metrics.items():
            mlflow.log_metric(f"test_{name}", value)
        
        # Log confusion matrix
        try:
            cm_fig = plot_confusion_matrix_heatmap(y_test_encoded, y_pred, le, street, 'RF')
            mlflow.log_figure(cm_fig, f"confusion_matrix_{street}_rf.png")
            plt.close(cm_fig)
        except Exception as e:
            print(f"      Warning: Could not plot confusion matrix: {e}")
        
        # Log ROC curves
        try:
            roc_fig, roc_auc_dict = plot_roc_curves(y_test_encoded, y_proba, le, street, 'RF')
            mlflow.log_figure(roc_fig, f"roc_curves_{street}_rf.png")
            plt.close(roc_fig)
        except Exception as e:
            print(f"      Warning: Could not plot ROC curves: {e}")
        
        # Log feature importance
        try:
            classifier = best_pipeline.named_steps['classifier']
            fi_fig, fi_df = plot_feature_importance(classifier, feature_names, street, 'RF')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_rf.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_rf.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        signature = infer_signature(sample_input, best_pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics}
        results.append({
            'pipeline': best_pipeline, 'calibrated_pipeline': final_pipeline,
            'label_encoder': le, 'model_type': 'RandomForest', 'y_proba': y_proba,
            'best_params': best_params, **all_metrics
        })
        print(f"      Test: F1={metrics['f1_weighted']:.3f}, Acc={metrics['accuracy']:.3f}, F1_macro={metrics['f1_macro']:.3f}")
    
    # =========================================================================
    # Model 3: Gradient Boosting Pipeline (NEW)
    # =========================================================================
    print("\n   Training GradientBoosting Pipeline (with hyperparameter tuning)...")
    with mlflow.start_run(run_name=f"{street}_GradientBoosting_Pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", "GradientBoosting")
        mlflow.log_param("pipeline", "StandardScaler + GradientBoosting")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("n_classes", len(le.classes_))
        mlflow.log_param("cv_folds", N_SPLITS)
        mlflow.log_param("hyperparam_tuning", True)
        mlflow.log_param("n_iter_search", N_ITER_RANDOM_SEARCH)
        
        base_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                validation_fraction=0.1,
                n_iter_no_change=10,  # Early stopping
                tol=1e-4
            ))
        ])
        
        # Conservative param grid - fewer options to reduce training time
        param_grid = {
            'classifier__n_estimators': [100, 150, 200],
            'classifier__max_depth': [3, 5, 7],
            'classifier__learning_rate': [0.05, 0.1, 0.15],
            'classifier__subsample': [0.8, 1.0],
            'classifier__min_samples_split': [2, 5],
            'classifier__min_samples_leaf': [1, 2],
        }
        
        best_pipeline, best_params, tuning_results = run_hyperparameter_tuning_with_logging(
            base_pipeline, param_grid, X_train, y_train_encoded, cv_tune, 'GB', street, feature_names, le
        )
        
        mlflow.log_params({f"best_{k}": str(v) for k, v in best_params.items()})
        mlflow.log_metric("tuning_best_score", tuning_results['best_score'])
        mlflow.log_dict(tuning_results, "hyperparameter_tuning_results.json")
        
        cv_metrics = run_cross_validation(best_pipeline, X_train, y_train_encoded, cv, 'GB')
        for name, value in cv_metrics.items():
            mlflow.log_metric(name, value)
        
        # Learning curve (conditionally - GB can be slow)
        lc_metrics = {}
        if COMPUTE_GB_LEARNING_CURVE:
            lc_fig, lc_metrics = plot_learning_curve(best_pipeline, X_train, y_train_encoded, cv, street, 'GB')
            if lc_fig:
                mlflow.log_figure(lc_fig, f"learning_curve_{street}_gb.png")
                for name, value in lc_metrics.items():
                    mlflow.log_metric(name, value)
                learning_curve_figures[f'GB_{street}'] = lc_fig
                plt.close(lc_fig)
        else:
            print("      Skipping GB learning curve (COMPUTE_GB_LEARNING_CURVE=False)")
        
        best_pipeline.fit(X_train, y_train_encoded)
        y_pred = best_pipeline.predict(X_test)
        y_proba = best_pipeline.predict_proba(X_test)
        
        metrics = calculate_all_metrics(y_test_encoded, y_pred, y_proba, le)
        for name, value in metrics.items():
            mlflow.log_metric(f"test_{name}", value)
        
        # Log confusion matrix
        try:
            cm_fig = plot_confusion_matrix_heatmap(y_test_encoded, y_pred, le, street, 'GB')
            mlflow.log_figure(cm_fig, f"confusion_matrix_{street}_gb.png")
            plt.close(cm_fig)
        except Exception as e:
            print(f"      Warning: Could not plot confusion matrix: {e}")
        
        # Log ROC curves
        try:
            roc_fig, roc_auc_dict = plot_roc_curves(y_test_encoded, y_proba, le, street, 'GB')
            mlflow.log_figure(roc_fig, f"roc_curves_{street}_gb.png")
            plt.close(roc_fig)
        except Exception as e:
            print(f"      Warning: Could not plot ROC curves: {e}")
        
        # Log feature importance
        try:
            classifier = best_pipeline.named_steps['classifier']
            fi_fig, fi_df = plot_feature_importance(classifier, feature_names, street, 'GB')
            if fi_fig:
                mlflow.log_figure(fi_fig, f"feature_importance_{street}_gb.png")
                mlflow.log_dict(fi_df.head(20).to_dict(), f"feature_importance_{street}_gb.json")
                plt.close(fi_fig)
        except Exception as e:
            print(f"      Warning: Could not log feature importance: {e}")
        
        signature = infer_signature(sample_input, best_pipeline.predict(X_train[:5]))
        mlflow.sklearn.log_model(best_pipeline, artifact_path="model", signature=signature)
        
        all_metrics = {**cv_metrics, **metrics, **lc_metrics}
        results.append({
            'pipeline': best_pipeline, 'label_encoder': le,
            'model_type': 'GradientBoosting', 'y_proba': y_proba,
            'best_params': best_params, **all_metrics
        })
        print(f"      Test: F1={metrics['f1_weighted']:.3f}, Acc={metrics['accuracy']:.3f}, F1_macro={metrics['f1_macro']:.3f}")
    
    best = max(results, key=lambda x: x.get('cv_f1_macro_mean', x['f1_macro']))
    return best, results, learning_curve_figures

print("   sklearn Pipeline training function ready")
print("   *** IMPORTANT: Pipelines bundle StandardScaler + Model together ***")
print("   *** At inference, pass RAW features - no manual scaling needed! ***")
print("   ")
print(f"   *** K-FOLD CROSS-VALIDATION: {N_SPLITS}-fold StratifiedKFold ***")
print(f"   *** HYPERPARAMETER TUNING: {N_ITER_RANDOM_SEARCH} trials, each logged to MLflow ***")
print(f"   *** LEARNING CURVES: {LEARNING_CURVE_POINTS} points (RF: {'enabled' if COMPUTE_RF_LEARNING_CURVE else 'disabled'}, GB: {'enabled' if COMPUTE_GB_LEARNING_CURVE else 'disabled'}) ***")
print("   *** PROBABILITY CALIBRATION: CalibratedClassifierCV for RF ***")
print("   *** FEATURE IMPORTANCE: Logged to each model run ***")
print("   *** ROC CURVES: Logged to each model run ***")
print("   ")
print("   *** CLASS IMBALANCE FIX: All models use class_weight='balanced' ***")
print("   ")
print("   Active models:")
print("   - LogisticRegression (L1/L2 regularization, tuned C)")
print("   - RandomForest (tuned hyperparameters, calibrated probabilities)")
print("   - GradientBoosting (tuned hyperparameters, early stopping)")

In [ ]:
# Train models per street with ROC curves, learning curves, and hyperparameter tuning
# UPDATED: Uses Pipeline (scaler bundled), saves scaling parameters
# UPDATED: Now captures learning curve figures from training function
print("\n[3/4] Training street-specific classifiers with Pipeline...")

best_models = {}
all_results = []
roc_figures = {}
all_learning_curves = {}  # Store learning curves for display

for street in STREETS:
    print(f"\n{'=' * 80}")
    print(f"STREET: {street.upper()}")
    print(f"{'=' * 80}")
    
    df = street_data[street]
    
    if len(df) < 100:
        print(f"   Skipping - only {len(df)} samples")
        continue
    
    # Get features for this street
    features = FEATURES_BY_STREET[street]
    available_features = [c for c in features if c in df.columns]
    
    # Prepare data
    df_clean = df.dropna(subset=['label_3class'])
    X = df_clean[available_features].copy()
    y = df_clean['label_3class']
    
    # Clean data: replace inf with nan, then fill nan with 0
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    
    # Clip extreme values to prevent overflow
    X = X.clip(lower=-1e9, upper=1e9)
    
    print(f"   Samples: {len(X):,}")
    print(f"   Features: {len(available_features)}")
    print(f"   Class distribution: {dict(y.value_counts())}")
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    print(f"   Train: {len(X_train):,} | Test: {len(X_test):,}")
    
    # Train pipelines - pass feature names for signature
    # Now returns 3 values: best, all_results, learning_curve_figures
    best, all_model_results, learning_curve_figures = train_sklearn_pipelines(
        X_train, X_test, y_train, y_test, street, available_features
    )
    
    # Store learning curves
    all_learning_curves.update(learning_curve_figures)
    
    print(f"\n   [BEST] {best['model_type']} -> F1: {best['f1_weighted']:.3f}, AUC: {best.get('roc_auc_ovr', 0):.3f}")
    if 'best_params' in best:
        print(f"   [BEST PARAMS] {best['best_params']}")
    
    # Get pipeline and label encoder from best model
    pipeline = best['pipeline']
    le = best['label_encoder']
    
    # Extract scaler from pipeline for saving scaling params
    scaler = pipeline.named_steps['scaler']
    
    # Encode test labels for evaluation
    y_test_encoded = le.transform(y_test)
    y_pred = pipeline.predict(X_test)
    
    print(f"\n   Classification Report:")
    print(classification_report(y_test_encoded, y_pred, target_names=le.classes_))
    
    # Plot and save ROC curves for best model
    try:
        fig, roc_auc_dict = plot_roc_curves(y_test_encoded, best['y_proba'], le, street, best['model_type'])
        roc_figures[street] = fig
        display(fig)
        plt.close(fig)
        print(f"   ROC AUC per class: {roc_auc_dict}")
    except Exception as e:
        print(f"   Warning: Could not plot ROC curves: {e}")
    
    # Store best model info (now stores pipeline instead of separate model/scaler)
    best_models[street] = {
        'pipeline': pipeline,
        'label_encoder': le,
        'model_type': best['model_type'],
        'best_params': best.get('best_params', {}),
        'test_f1': best['f1_weighted'],
        'test_accuracy': best['accuracy'],
        'test_precision': best['precision_weighted'],
        'test_recall': best['recall_weighted'],
        'test_roc_auc': best.get('roc_auc_ovr', 0),
        'cv_f1_macro_mean': best.get('cv_f1_macro_mean', 0),
        'cv_f1_macro_std': best.get('cv_f1_macro_std', 0),
        'lc_train_val_gap': best.get('lc_train_val_gap', 0),
        'features': available_features
    }
    
    # Track results for summary (include hyperparameter tuning and learning curve metrics)
    all_results.append({
        'street': street,
        'model_type': best['model_type'],
        'best_params': str(best.get('best_params', {})),
        'test_f1_weighted': float(best['f1_weighted']),
        'test_f1_macro': float(best['f1_macro']),
        'test_accuracy': float(best['accuracy']),
        'test_precision_weighted': float(best['precision_weighted']),
        'test_recall_weighted': float(best['recall_weighted']),
        'test_roc_auc_ovr': float(best.get('roc_auc_ovr', 0)),
        'cv_f1_macro_mean': float(best.get('cv_f1_macro_mean', 0)),
        'cv_f1_macro_std': float(best.get('cv_f1_macro_std', 0)),
        'lc_final_train_score': float(best.get('lc_final_train_score', 0)),
        'lc_final_val_score': float(best.get('lc_final_val_score', 0)),
        'lc_train_val_gap': float(best.get('lc_train_val_gap', 0)),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'n_features': len(available_features)
    })
    
    # ========================================================================
    # Register best PIPELINE in MLflow Model Registry
    # Pipeline includes scaler - webapp can pass RAW features directly
    # ========================================================================
    model_name = f"{MODEL_REGISTRY_PREFIX}.01-opponent-modeling-{street}-v3"
    
    with mlflow.start_run(run_name=f"{street}_best_pipeline"):
        mlflow.log_param("street", street)
        mlflow.log_param("model_type", best['model_type'])
        mlflow.log_param("pipeline", "StandardScaler + " + best['model_type'])
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", len(available_features))
        mlflow.log_param("features", str(available_features))
        mlflow.log_param("classes", str(list(le.classes_)))
        
        # Log best hyperparameters
        if 'best_params' in best:
            for k, v in best['best_params'].items():
                mlflow.log_param(f"best_{k}", str(v))
        
        # Log all metrics
        mlflow.log_metric("test_accuracy", best['accuracy'])
        mlflow.log_metric("test_f1_weighted", best['f1_weighted'])
        mlflow.log_metric("test_f1_macro", best['f1_macro'])
        mlflow.log_metric("test_precision_weighted", best['precision_weighted'])
        mlflow.log_metric("test_precision_macro", best['precision_macro'])
        mlflow.log_metric("test_recall_weighted", best['recall_weighted'])
        mlflow.log_metric("test_recall_macro", best['recall_macro'])
        if 'roc_auc_ovr' in best:
            mlflow.log_metric("test_roc_auc_ovr", best['roc_auc_ovr'])
        if 'roc_auc_ovo' in best:
            mlflow.log_metric("test_roc_auc_ovo", best['roc_auc_ovo'])
        
        # Log CV metrics
        if 'cv_f1_macro_mean' in best:
            mlflow.log_metric("cv_f1_macro_mean", best['cv_f1_macro_mean'])
            mlflow.log_metric("cv_f1_macro_std", best['cv_f1_macro_std'])
        
        # Log learning curve metrics
        if 'lc_train_val_gap' in best:
            mlflow.log_metric("lc_train_val_gap", best['lc_train_val_gap'])
            mlflow.log_metric("lc_final_train_score", best.get('lc_final_train_score', 0))
            mlflow.log_metric("lc_final_val_score", best.get('lc_final_val_score', 0))
        
        # Create signature with RAW input (pipeline handles scaling)
        sample_input = pd.DataFrame(X_train[:5].values, columns=available_features)
        signature = infer_signature(sample_input, pipeline.predict(X_train[:5]))
        
        # Log the PIPELINE (includes scaler) to Unity Catalog registry
        mlflow.sklearn.log_model(
            pipeline,  # Pipeline, not just model
            artifact_path="model",
            signature=signature,
            registered_model_name=model_name
        )
        
        # ====================================================================
        # SAVE SCALING PARAMETERS as artifact (for debugging/backup)
        # ====================================================================
        scaling_params = {
            'feature_names': available_features,
            'means': scaler.mean_.tolist(),
            'stds': scaler.scale_.tolist(),
            'street': street,
            'model_type': best['model_type'],
            'best_params': best.get('best_params', {}),
            'model_version': 'v3',
            'note': 'Pipeline includes scaler - these params are for reference only'
        }
        mlflow.log_dict(scaling_params, "scaling_params.json")
        
        # Log ROC curve figure if available
        if street in roc_figures:
            mlflow.log_figure(roc_figures[street], f"roc_curves_{street}.png")
        
        print(f"   Registered PIPELINE in MLflow: {model_name}")
        print(f"   *** Pipeline includes StandardScaler - pass RAW features at inference ***")
    
    # Save model metadata + scaling params to UC Volume
    model_metadata = {
        'model_type': best['model_type'],
        'pipeline': 'StandardScaler + ' + best['model_type'],
        'best_params': str(best.get('best_params', {})),
        'features': available_features,
        'scaling_means': scaler.mean_.tolist(),
        'scaling_stds': scaler.scale_.tolist(),
        'test_f1_weighted': float(best['f1_weighted']),
        'test_f1_macro': float(best['f1_macro']),
        'test_accuracy': float(best['accuracy']),
        'test_precision_weighted': float(best['precision_weighted']),
        'test_recall_weighted': float(best['recall_weighted']),
        'test_roc_auc_ovr': float(best.get('roc_auc_ovr', 0)),
        'cv_f1_macro_mean': float(best.get('cv_f1_macro_mean', 0)),
        'cv_f1_macro_std': float(best.get('cv_f1_macro_std', 0)),
        'lc_train_val_gap': float(best.get('lc_train_val_gap', 0)),
        'classes': le.classes_.tolist(),
        'n_train': len(X_train),
        'n_test': len(X_test),
        'note': 'Pipeline includes scaler - scaling params saved for reference'
    }
    
    metadata_df = spark.createDataFrame([model_metadata])
    metadata_path = f"{MODELS_DIR}sp3_{street}_metadata"
    metadata_df.write.mode('overwrite').json(metadata_path)
    print(f"   Saved metadata + scaling params: {metadata_path}")

# Save all results summary to UC Volume
if all_results:
    results_df = spark.createDataFrame(all_results)
    results_df.write.mode('overwrite').parquet(f"{MODELS_DIR}sp3_model_comparison")
    print(f"\n   Saved results comparison: {MODELS_DIR}sp3_model_comparison")

print("\n" + "=" * 80)
print("IMPORTANT: Models are now sklearn Pipelines with scaler included!")
print("At inference, pass RAW features directly - no manual scaling needed.")
print("=" * 80)

In [ ]:
# Feature Importance Analysis
# UPDATED: Extract classifier from pipeline to get feature importances
print("\n[4/4] Feature Importance Analysis...")

for street in STREETS:
    if street not in best_models:
        continue

    model_info = best_models[street]
    pipeline = model_info['pipeline']
    features = model_info['features']
    model_type = model_info['model_type']

    # Extract the classifier from the pipeline
    classifier = pipeline.named_steps['classifier']

    print(f"\n   {street.upper()} - {model_type} Feature Importances:")

    # Get feature importances based on model type
    importances = None

    if hasattr(classifier, 'feature_importances_'):
        # Works for RandomForest, GradientBoosting, and HistGradientBoosting
        importances = classifier.feature_importances_
        print(f"      (using feature_importances_ attribute)")
    elif hasattr(classifier, 'coef_'):
        # For LogisticRegression, use mean absolute coefficient across classes
        importances = np.abs(classifier.coef_).mean(axis=0)
        print(f"      (using mean absolute coefficients)")
    
    if importances is None:
        print(f"      No feature importances available for {model_type}")
        continue
    
    # Validate lengths match
    if len(importances) != len(features):
        print(f"      WARNING: Feature count mismatch - {len(features)} features vs {len(importances)} importances")
        # Use minimum length to avoid index errors
        min_len = min(len(features), len(importances))
        features = features[:min_len]
        importances = importances[:min_len]
    
    # Sort by importance
    feature_importance = sorted(
        zip(features, importances),
        key=lambda x: x[1],
        reverse=True
    )
    
    # Display top 15 features
    print(f"      Top 15 features:")
    for feat, imp in feature_importance[:15]:
        print(f"         {feat:40s}: {imp:.4f}")
    
    # Create a bar chart for feature importances
    try:
        fig, ax = plt.subplots(figsize=(12, 8))
        top_n = min(20, len(feature_importance))
        top_features = feature_importance[:top_n]
        
        y_pos = np.arange(top_n)
        ax.barh(y_pos, [imp for _, imp in top_features], align='center', color='steelblue')
        ax.set_yticks(y_pos)
        ax.set_yticklabels([feat for feat, _ in top_features])
        ax.invert_yaxis()  # Top feature at top
        ax.set_xlabel('Importance', fontsize=12)
        ax.set_title(f'Feature Importance - {street.upper()} ({model_type})', fontsize=14)
        ax.grid(True, alpha=0.3, axis='x')
        
        plt.tight_layout()
        display(fig)
        plt.close(fig)
    except Exception as e:
        print(f"      Could not plot feature importance chart: {e}")
    
    # Log to MLflow
    try:
        with mlflow.start_run(run_name=f"{street}_feature_importance"):
            mlflow.log_param("street", street)
            mlflow.log_param("model_type", model_type)
            for i, (feat, imp) in enumerate(feature_importance[:20]):
                mlflow.log_metric(f"importance_{i+1}", float(imp))
                mlflow.log_param(f"feature_{i+1}", feat)
            
            # Log the feature importance figure
            if 'fig' in dir():
                mlflow.log_figure(fig, f"feature_importance_{street}.png")
    except Exception as e:
        print(f"      Could not log to MLflow: {e}")

In [ ]:
# Summary
elapsed = time.time() - start_time
print("\n" + "=" * 80)
print("SP-3 Opponent Modeling COMPLETE!")
print("=" * 80)
print(f"\nRuntime: {elapsed/60:.1f} minutes")
print(f"Models trained: {len(best_models)} streets")

print("\n" + "-" * 80)
print("BEST MODELS:")
print("-" * 80)
for street in STREETS:
    if street in best_models:
        info = best_models[street]
        print(f"   {street}: {info['model_type']}")
        print(f"      F1={info['test_f1']:.4f}, Acc={info['test_accuracy']:.4f}")
        print(f"      Precision={info['test_precision']:.4f}, Recall={info['test_recall']:.4f}")
        print(f"      ROC AUC={info['test_roc_auc']:.4f}")

print("\n" + "-" * 80)
print("MODEL CONFIGURATION:")
print("-" * 80)
print("   LogisticRegression: L2 regularization (C=1.0), multinomial")
print("   RandomForest: 200 trees, max_depth=15, class_weight='balanced'")
print("   GradientBoosting: 200 trees, max_depth=7, subsample=0.8")
print("   HistGradientBoosting: 200 iter, max_depth=10, L2=0.1, early_stopping")

print("\n" + "-" * 80)
print("OUTPUT - MLflow Model Registry:")
print("-" * 80)
for street in STREETS:
    if street in best_models:
        print(f"   - sp3_opponent_{street}")

print("\n" + "-" * 80)
print("OUTPUT - UC Volume:")
print("-" * 80)
for street in STREETS:
    if street in best_models:
        print(f"   - {MODELS_DIR}sp3_{street}_metadata/")
print(f"   - {MODELS_DIR}sp3_model_comparison/")

print("\n" + "-" * 80)
print("DATA LINEAGE:")
print("-" * 80)
print(f"   Input: SP-2 output ({INPUT_PATH})")
print(f"   Output: SP-3 opponent models (MLflow Registry)")

print("\n" + "-" * 80)
print("METRICS LOGGED TO MLFLOW:")
print("-" * 80)
print("   Classification: accuracy, f1_weighted, f1_macro, precision_weighted,")
print("                   precision_macro, recall_weighted, recall_macro")
print("   ROC: roc_auc_ovr, roc_auc_ovo")
print("   Artifacts: ROC curve plots (roc_curves_{street}.png)")

print("\n" + "-" * 80)
print("USAGE EXAMPLE:")
print("-" * 80)
print("   # Load model from MLflow")
print("   import mlflow")
print("   model = mlflow.sklearn.load_model('models:/sp3_opponent_preflop/latest')")
print("   ")
print("   # Make predictions")
print("   y_pred = model.predict(X_new)")

print(f"\n[SUCCESS] Opponent modeling complete!")

In [ ]:
# ============================================================================
# [5/5] GENERATE PREDICTIONS ON FULL DATASET AND SAVE
# UPDATED: Uses Pipeline (no separate scaler needed)
# This allows notebook 04 to load pre-computed predictions without loading models
# ============================================================================
print("\n[5/5] Generating predictions on full dataset...")

from pyspark.sql.window import Window

# Reload full dataset (we sampled for training, now predict on all)
full_df = spark.read.parquet(INPUT_PATH)
full_count = full_df.count()
print(f"   Full dataset: {full_count:,} rows")

# ============================================================================
# CRITICAL: Create hand-level idx BEFORE filtering by street
# This ensures idx is consistent across the entire hand (1, 2, 3, ... N)
# NOT per-street (which would restart at 1 for each street)
# ============================================================================
print("   Creating hand-level idx (action sequence across entire hand)...")
hand_action_window = Window.partitionBy('hand_id').orderBy(F.monotonically_increasing_id())
full_df = full_df.withColumn('idx', F.row_number().over(hand_action_window))

# Verify idx is hand-level
print(f"\n[TRACE] Verifying hand-level idx for {TRACE_HAND_ID}:")
full_df.filter(F.col('hand_id') == TRACE_HAND_ID).select(
    'hand_id', 'idx', 'actor', 'street', 'action_type'
).orderBy('idx').show(20, truncate=False)

# Class mappings
CLASS_MAPPING = {0: 'air', 1: 'middle', 2: 'nutted'}
STRENGTH_MAPPING = {'air': 0.2, 'middle': 0.5, 'nutted': 0.8}

all_predictions = []

for street in STREETS:
    if street not in best_models:
        print(f"   Skipping {street} - no model available")
        continue
    
    print(f"\n   Processing {street}...")
    
    model_info = best_models[street]
    pipeline = model_info['pipeline']  # Pipeline includes scaler
    le = model_info['label_encoder']
    features = model_info['features']
    
    # Get street data - idx is already set at hand level
    street_spark = full_df.filter(F.col('street') == street)
    street_count = street_spark.count()
    
    if street_count == 0:
        print(f"      No data for {street}")
        continue
    
    print(f"      {street_count:,} rows")
    
    # Select columns - idx is already created at hand level
    cols_to_select = ['hand_id', 'idx', 'actor'] + [f for f in features if f in street_spark.columns]
    street_pandas = street_spark.select(cols_to_select).toPandas()
    
    # Prepare features
    available_features = [f for f in features if f in street_pandas.columns]
    X = street_pandas[available_features].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    X = X.clip(lower=-1e9, upper=1e9)
    
    # Predict using Pipeline (handles scaling internally)
    # No need to call scaler.transform() - pipeline does it automatically
    predictions = pipeline.predict(X)
    
    # Map to bucket names and strengths
    street_pandas['predicted_bucket'] = [CLASS_MAPPING.get(int(p), 'middle') for p in predictions]
    street_pandas['predicted_strength'] = street_pandas['predicted_bucket'].map(STRENGTH_MAPPING)
    street_pandas['street'] = street
    
    # Keep only key columns
    pred_df = street_pandas[['hand_id', 'idx', 'actor', 'street', 'predicted_bucket', 'predicted_strength']]
    all_predictions.append(pred_df)
    
    print(f"      Distribution: {pred_df['predicted_bucket'].value_counts().to_dict()}")

# Combine all predictions
if all_predictions:
    combined_predictions = pd.concat(all_predictions, ignore_index=True)
    print(f"\n   Total predictions: {len(combined_predictions):,}")
    
    # Verify no duplicate (hand_id, idx) combinations
    dup_check = combined_predictions.groupby(['hand_id', 'idx']).size()
    dups = dup_check[dup_check > 1]
    if len(dups) > 0:
        print(f"   WARNING: Found {len(dups)} duplicate (hand_id, idx) pairs!")
        print(f"   This could cause row explosion in notebook 04 joins")
    else:
        print(f"   ✓ No duplicate (hand_id, idx) pairs - join will be clean")
    
    # Convert to Spark and save
    predictions_spark = spark.createDataFrame(combined_predictions)
    
    # Save predictions to UC Volume
    PREDICTIONS_OUTPUT_PATH = UC_VOLUME_DIR + 'processed/sp3_opponent_predictions'
    predictions_spark.write.mode('overwrite').parquet(PREDICTIONS_OUTPUT_PATH)
    print(f"   Saved predictions to: {PREDICTIONS_OUTPUT_PATH}")
    
    # TRACE: Show predictions for trace hand (should have unique idx per row)
    print(f"\n[TRACE] Predictions for {TRACE_HAND_ID} (idx should be unique per row):")
    predictions_spark.filter(F.col('hand_id') == TRACE_HAND_ID).orderBy('idx').show(20, truncate=False)
else:
    print("   ERROR: No predictions generated!")

print("\n" + "=" * 80)
print("SP-3 COMPLETE - Predictions saved for notebook 04")
print("=" * 80)
print("*** Models are sklearn Pipelines with StandardScaler included ***")
print("*** At inference, pass RAW features - no manual scaling needed ***")

---
# ARCHIVED: Original sklearn Implementation

The cells below contain the original sklearn-based implementation.
This required sampling for large datasets. The SparkML version above
can handle the full dataset distributed across the cluster.

---

In [ ]:
# # ARCHIVED: sklearn imports
# import pandas as pd
# import numpy as np
# from pathlib import Path
# from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
# from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score
# import joblib
# import json

In [ ]:
# # ARCHIVED: sklearn data loading (required sampling)
# # Load data from Spark Parquet
# # print("\n[1/3] Loading labeled data...")
# # 
# # spark_df = spark.read.parquet(parquet_path)
# # total_rows = spark_df.count()
# # 
# # # Sample if too large for pandas (sklearn can't handle 4M+ rows)
# # MAX_ROWS_FOR_TRAINING = 500000
# # if total_rows > MAX_ROWS_FOR_TRAINING:
# #     sample_fraction = MAX_ROWS_FOR_TRAINING / total_rows
# #     spark_df = spark_df.sample(fraction=sample_fraction, seed=42)
# # 
# # # Convert to pandas for sklearn
# # df = spark_df.toPandas()

In [ ]:
# # ARCHIVED: sklearn model definitions
# # def build_preprocessor(num_cols):
# #     numeric_transformer = Pipeline([
# #         ('impute', SimpleImputer(strategy='median')),
# #         ('scale', StandardScaler()),
# #     ])
# #     categorical_transformer = Pipeline([
# #         ('impute', SimpleImputer(strategy='most_frequent')),
# #         ('encode', OneHotEncoder(handle_unknown='ignore')),
# #     ])
# #     return ColumnTransformer([
# #         ('num', numeric_transformer, num_cols),
# #         ('cat', categorical_transformer, CATEGORICAL_COLS),
# #     ])
# # 
# # MODEL_SEARCH_SPACES = {
# #     'HistGradientBoosting': {
# #         'estimator': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
# #         'param_distributions': {
# #             'clf__max_depth': [3, 4, 5],
# #             'clf__learning_rate': [0.05, 0.1],
# #             'clf__max_iter': [200, 300],
# #             'clf__min_samples_leaf': [20, 40],
# #         },
# #         'n_iter': 8,
# #     },
# # }

In [ ]:
# # ARCHIVED: sklearn training loop
# # for street in STREETS:
# #     features = FEATURES_BY_STREET[street]
# #     street_df = df[df['street'] == street].dropna(subset=required_cols).copy()
# #     
# #     X = street_df[features + CATEGORICAL_COLS]
# #     y = street_df[TARGET_COL]
# #     
# #     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y)
# #     
# #     pipeline = Pipeline([
# #         ('prep', build_preprocessor(features)),
# #         ('clf', HistGradientBoostingClassifier(random_state=RANDOM_STATE, **param_dict))
# #     ])
# #     pipeline.fit(X_train, y_train)
# #     y_pred = pipeline.predict(X_test)
# #     
# #     test_f1 = f1_score(y_test, y_pred, average='weighted')